# 📝 `read_todos()` 工具的设计意图

这是个特别值得搞清楚的设计模式。`read_todos` 看起来很"多余"(我都写进去了,为啥还要读回来?),但它正是 deep agent 处理长任务的核心招数。

本笔记是对 [1_todo.ipynb](./1_todo.ipynb) 中 `read_todos` 工具的深度解读。

## 一、先理清"两个地方都能存信息"

LangGraph 的 agent 里有两块独立的"存储":

| 存储位置 | 是什么 | LLM 看得到吗? |
|---|---|---|
| **`state["todos"]`** | TypedDict 的一个字段,**权威数据**,只有最新一份 | ❌ **看不到**(state 不会自动塞进 prompt) |
| **`state["messages"]`** | 完整对话历史,append-only,所有 ToolMessage / AIMessage 都在里面 | ✅ **每轮都重放给 LLM** |

> 这是关键:**LLM 唯一的"视线"就是 messages**。state 里的 `todos` 字段对它来说是不可见的,除非有东西把它**搬进 messages**。

## 二、`write_todos` 做了什么?

```python
return Command(update={
    "todos": todos,                                              # ← 写进 state
    "messages": [ToolMessage(f"Updated todo list to {todos}", ...)]  # ← 同时塞一条进 messages
})
```

它**同时做两件事**:
1. 把新 todo 列表写进 `state["todos"]`(权威存储)
2. 往 messages 追加一条 `ToolMessage`,内容是 "Updated todo list to [...]" —— 这是给 LLM 看的

所以**当下这一刻**,LLM 是知道 todo 列表的(就在 messages 末尾)。

## 三、为什么还要 `read_todos`?—— 核心动机:**对抗 context rot**

想象 agent 跑了 20 轮:

```
messages = [
  Human: "调研 MCP 协议"
  AI: write_todos(["搜MCP", "总结", "对比其他"])
  Tool: "Updated todo list to [...]"     ← 计划在这里!
  AI: web_search("MCP")
  Tool: <一大段搜索结果>
  AI: web_search("Anthropic MCP spec")
  Tool: <又一大段>
  AI: web_search("...")
  Tool: <又一大段>
  ... (15 轮 tool 调用,每个返回都很长)
  AI: ???                                 ← 此刻 LLM 在想"我接下来该干嘛?"
]
```

**问题**:那条 `Updated todo list to [...]` 已经被埋在 messages **最上面**,前面隔着十几条又长又乱的搜索结果。LLM 在做下一步决策时:
- 计划离当前位置太远,**注意力衰减**(这就是 "context rot")
- LLM 可能开始**跑题**、忘记原计划、重复已完成的步骤

**`read_todos` 的解决办法**:让 LLM **主动调用**一次"读 todo",于是 messages 末尾**新增**一条:

```
Tool: "Current TODO List:
       1. ✅ 搜MCP (completed)
       2. 🔄 总结 (in_progress)
       3. ⏳ 对比其他 (pending)"
```

现在 LLM 在做下一步决策时,**最新一条消息就是计划**,注意力一下子被拉回到任务上。

> 这就是 Manus 博客里说的 **"recitation"(背诵)** 模式:让 agent 周期性地"复述"目标,把计划**推到 context 的最近端**,防止跑偏。

## 四、为什么 `read_todos` 要从 state 读,而不是从 messages 里翻?

因为 messages 里可能有**多条** "Updated todo list to [...]"(每次 `write_todos` 都加一条),而:
- 它们都是历史快照,中间状态会让 LLM 困惑
- LLM 不知道哪个是"最新的"
- 直接从 `state["todos"]` 读 → **永远是当前的权威版本**

`state` 就像数据库,`messages` 就像聊天记录。要查"现在的状态",当然问数据库,不是翻聊天记录。

## 五、这个工具的"特殊性":它本质上是个"自我提示"

普通工具是为了**获取外部信息**(web_search、calculator 等)。

而 `read_todos` 不获取任何新信息 —— 它读的就是 agent **自己几轮前写下的东西**。它的真正作用是:**让 LLM 把自己的计划"复制粘贴"到当前对话末尾,刷新自己的注意力**。

回头看 `TODO_USAGE_INSTRUCTIONS` 里的指令:

```
1. write_todos 创建计划
2. After you accomplish a TODO, use the read_todos to read the TODOs   ← 每完成一项都要"读"一次
   in order to remind yourself of the plan.
3. Reflect on what you've done and the TODO.
4. Mark you task as completed, and proceed to the next TODO.
```

整个流程其实是一个**写-做-读-反思-改状态-做**的循环。`read_todos` 在第 2 步登场,作用是"做完一件事后,先把计划再看一遍,确认自己没跑偏"。

## 一句话总结

> **`read_todos` 不是给程序员读的,是给 LLM 自己读的。** 它从权威存储 `state["todos"]` 里取出最新计划,把它**重新塞回 messages 末尾**,刷新 LLM 的注意力,对抗长任务中的 context rot —— 这是 deep agent "推荐自己复述目标"的工程手段。